#### 문항 1. 금융위 주식시세정보 API 호출과 오류 처리

공공데이터포털에서 발급받은 서비스키로 금융위원회 주식시세정보 API를 호출하시오.

* 대상: https://www.data.go.kr/data/15094808/openapi.do

* 서비스키는 .env로 분리하고, .gitignore에 .env를 추가할 것

* 코드에 키를 하드코딩하지 않고 불러와 사용할 것

* 삼성전자(005930) 최근 5영업일 시세를 조회해 JSON을 dict로 파싱할 것

* 다음 세 가지 오류 상황을 각각 구분해 처리할 것

    1. 인증키 오류 (SERVICE_KEY_IS_NOT_REGISTERED_ERROR)

    2. 일일 쿼터 초과 (LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR)

    3. 필수 파라미터 누락 (INVALID_REQUEST_PARAMETER_ERROR)

* .env.example 파일을 함께 제출할 것 (키 값은 비울 것)

In [10]:
import requests

BASE_URL = ("https://apis.data.go.kr/1160100/service/"
            "GetStockSecuritiesInfoService")

In [19]:
class ApiError(Exception):
    pass

In [20]:
import requests
from urllib.parse import urlencode

class GetStockAPI:
    """공공데이터 금융위원회 주식시세정보 API"""

    def __init__(self, key, **kwargs):
        self.key = key 
        self.extra_params = {
            "resultType": "json",
            **kwargs,
        }

    def _fetch(self, url, params):
        query = urlencode(params)
        full_url = f"{url}?serviceKey={self.key}&{query}"

        r = requests.get(full_url, timeout=10)
        r.raise_for_status()

        try:
            body = r.json()
        except ValueError:
            raise ApiError(f"JSON이 아닌 응답: {r.text[:200]}")

        header = body["response"]["header"]
        code, msg = header["resultCode"], header["resultMsg"]

        if code != "00":
            if code == '30' or "SERVICE_KEY_IS_NOT_REGISTERED_ERROR" in msg:
                raise ApiError("인증키 오류 — 승인이 반영됐는지 확인하세요.")
            if code == '22' or "LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR" in msg:
                raise ApiError("일일 쿼터(10,000건)를 초과했습니다. 내일 다시 시도하세요.")
            if code == '10' or "INVALID_REQUEST_PARAMETER_ERROR" in msg:
                raise ApiError(f"필수 파라미터 누락 — 전달값: {list(params)}")
            raise ApiError(f"[{code}] {msg}")

        return body["response"]["body"]

    def get_stocks(self, page, size):
        url = BASE_URL + '/getStockPriceInfo'
        params = {**self.extra_params, 'pageNo': page, 'numOfRows': size}
        body = self._fetch(url, params)
        return body["items"]["item"]

In [21]:
samsung = GetStockAPI(KEY, likeSrtnCd="005930")

try:
    items = samsung.get_stocks(1, 5)
    for it in items:
        print(it["basDt"], it["itmsNm"], it["clpr"], it["trqu"])
except ApiError as e:
    print("API 오류:", e)

20260827 삼성전자 266000 16829395
20260826 삼성전자 261500 19532523
20260825 삼성전자 257000 21617407
20260824 삼성전자 257000 32451940
20260821 삼성전자 281500 27746471


#### 문항 2. DB 스키마 설계와 적재 검증

수집한 원본을 보관할 테이블을 만들고, 가공 없이 적재하시오. (과제1의 코드를 활용하여 진행)

1. MariaDB에 fsc_db 데이터베이스를 만들고 전용 계정으로 접속할 것

2. 테이블 raw_item을 다음 컬럼으로 생성할 것 raw_id / source / url / collected_at / payload / content_hash

    * content_hash에 UNIQUE 제약

    * (source, collected_at) 복합 인덱스 설정

3. 다음 10개 종목의 최근 2025년 주식 시세를 수집하여 source='fsc_api' 로, 원문 JSON 그대로 payload에 적재할 것
```
CODES = ["005930", "000660", "035420", "051910", "005380",
         "006400", "035720", "068270", "105560", "055550"]
```

4. executemany 배치 삽입 + ON DUPLICATE KEY UPDATE 를 쓸 것

5. 같은 스크립트를 두 번 실행하고, 행 수가 변하지 않음을 SQL로 확인할 것

6. DB 비밀번호도 .env로 관리할 것

참고용 해싱 코드
```
import hashlib

key_src = '해싱하여 중복되는 값이 들어올 경우 UNIQUE 제약조건을 발생시킬 문자열'
# 예시) fsc_api|20250101|005930
# 수집소스|날짜|주식코드를 해싱하여 같은 값이 또 들어올 경우 Duplicate 처리
hashlib.sha256(key_src.encode()).hexdigest()
```

In [ ]:
import os, time, json, hashlib
import pandas as pd
import pymysql
from datetime import datetime
from dotenv import load_dotenv
from finance_stock import GetStockAPI 

load_dotenv()
KEY = os.getenv("DATA_GO_KR_KEY")

CODES = ["005930", "000660", "035420", "051910", "005380",
         "006400", "035720", "068270", "105560", "055550"]
BEGIN, END = "20250101", "20251231"
NUM_OF_ROWS = 300
DELAY = 0.3
SOURCE = 'fsc_api'
HASHING_COLUMNS = ["basDt", "srtnCd"]
URL = ("https://apis.data.go.kr/1160100/service/"
       "GetStockSecuritiesInfoService/getStockPriceInfo")
BATCH = 500

SQL = """
INSERT INTO raw_item (source, url, collected_at, payload, content_hash)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    payload      = VALUES(payload),
    collected_at = VALUES(collected_at)
"""

def connect():
    return pymysql.connect(
        host=os.getenv("DB_HOST", "localhost"),
        user=os.getenv("DB_USER", "analyzer"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME", "fsc_db"),
        charset="utf8mb4",
    )

def status():
    q = """
    SELECT source, COUNT(*) AS 건수, COUNT(DISTINCT content_hash) AS 고유건수,
           MIN(collected_at) AS 최초수집, MAX(collected_at) AS 최종수집
    FROM raw_item GROUP BY source
    """
    conn = connect()
    try:
        return pd.read_sql(q, conn)
    finally:
        conn.close()

def main():
    result = []
    for code in CODES:
        api = GetStockAPI(KEY, likeSrtnCd=code, beginBasDt=BEGIN, endBasDt=END)
        result.extend(api.get_stocks(1, NUM_OF_ROWS))
        time.sleep(DELAY) 

    df = pd.DataFrame(result)
    collected_dt = datetime.now()
    rows = []
    for rec in df.to_dict(orient='records'):
        payload = json.dumps(rec, ensure_ascii=False)
        key_str = '|'.join([SOURCE] + [rec[c] for c in HASHING_COLUMNS])
        hash_key = hashlib.sha256(key_str.encode()).hexdigest()
        rows.append((SOURCE, URL, collected_dt, payload, hash_key))

    print("=== 적재 전 ===\n", status())

    conn = connect()
    try:
        for i in range(0, len(rows), BATCH):
            with conn.cursor() as cur:
                cur.executemany(SQL, rows[i:i + BATCH])
            conn.commit()
    finally:
        conn.close()

    print("\n=== 적재 후 ===\n", status())

if __name__ == "__main__":
    main()